In [3]:
!pip install -U pip setuptools wheel
!pip install jupyter pandas numpy matplotlib seaborn scikit-learn imbalanced-learn xgboost lightgbm imbalanced-learn 
!pip install kagglehub

In [4]:
#To download datset from Kaggle:
import kagglehub
from pathlib import Path
import pandas as pd
path = kagglehub.dataset_download("dartweichen/student-life") 
print("Path to dataset files:", path)

Path to dataset files: /Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1


In [23]:
#Base path for dataset files:
from pathlib import Path

base_path = Path(path)
print(base_path)

/Users/leilatawfik/.cache/kagglehub/datasets/dartweichen/student-life/versions/1


In [24]:
#Loading the stress response data (label):

stress_dir = base_path / "dataset" / "EMA" / "response" / "Stress"
print("stress_dir exists?", stress_dir.exists())
print("json count:", len(list(stress_dir.rglob("*.json"))))
print("csv count:", len(list(stress_dir.rglob("*.csv"))))

stress_dir exists? True
json count: 49
csv count: 0


In [8]:
import json

all_dfs = []
skipped = 0

for file in stress_dir.rglob("*.json"):
    try:
        text = file.read_text(encoding="utf-8", errors="ignore").strip()
        if not text:
            skipped += 1
            continue

        try:
            obj = json.loads(text)
            df = pd.json_normalize(obj if isinstance(obj, list) else [obj])
        except json.JSONDecodeError:
            # JSON lines fallback
            rows = [json.loads(line) for line in text.splitlines() if line.strip()]
            df = pd.json_normalize(rows) if rows else pd.DataFrame()

        if df.empty:
            skipped += 1
            continue

        df["student_id"] = file.stem
        df["source_file"] = file.name
        all_dfs.append(df)

    except Exception:
        skipped += 1

stress_df = pd.concat(all_dfs, ignore_index=True, sort=False)

print("✅ stress_df shape:", stress_df.shape)
print("✅ participants:", stress_df["student_id"].nunique())
print("✅ skipped files:", skipped)
print("Columns:", list(stress_df.columns))
print(stress_df.head())

✅ stress_df shape: (2408, 6)
✅ participants: 49
✅ skipped files: 0
Columns: ['null', 'resp_time', 'level', 'location', 'student_id', 'source_file']
                       null   resp_time level location  student_id  \
0  43.70477575,-72.28844073  1364121982   NaN      NaN  Stress_u44   
1                         2  1364121983   NaN      NaN  Stress_u44   
2  43.70637091,-72.28704334  1364118696   NaN      NaN  Stress_u44   
3                         4  1364121980   NaN      NaN  Stress_u44   
4                         3  1364121985   NaN      NaN  Stress_u44   

       source_file  
0  Stress_u44.json  
1  Stress_u44.json  
2  Stress_u44.json  
3  Stress_u44.json  
4  Stress_u44.json  


In [9]:
stress_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2408 entries, 0 to 2407
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   null         240 non-null    str  
 1   resp_time    2408 non-null   int64
 2   level        2167 non-null   str  
 3   location     2167 non-null   str  
 4   student_id   2408 non-null   str  
 5   source_file  2408 non-null   str  
dtypes: int64(1), str(5)
memory usage: 113.0 KB


In [10]:
# Number of missing values in 'level'
missing_count = stress_df["level"].isna().sum()
total_count = len(stress_df["level"])
print(total_count)

# Percentage of missing values in stress level:
print(missing_count/total_count * 100)

print("Missing values in 'level':", missing_count)

2408
10.008305647840531
Missing values in 'level': 241


In [11]:
def read_sensing_csv(filepath, encoding="utf-8", sniff_lines=50):
    """
    Reads sensing CSVs robustly.
    - If no trailing comma issue is detected, uses pd.read_csv normally.
    - If trailing commas / inconsistent trailing empty fields are detected,
      reads as text and normalizes each row to the header length.
    """
    # --- Detect trailing-comma pattern quickly (header or first N lines)
    with open(filepath, "r", encoding=encoding, errors="replace") as f:
        lines = f.read().splitlines()

    if not lines:
        return pd.DataFrame()

    header_line = lines[0]
    sample_lines = lines[1:1+sniff_lines]

    def has_trailing_comma(s: str) -> bool:
        return s.rstrip().endswith(",")

    suspicious = has_trailing_comma(header_line) or any(has_trailing_comma(s) for s in sample_lines)

    # --- If not suspicious, just do the normal pandas read
    if not suspicious:
        df = pd.read_csv(filepath)
        df.columns = df.columns.astype(str).str.strip()
        return df

    # --- Robust path: normalize row lengths to header length
    header = [h.strip() for h in header_line.split(",")]

    # remove trailing empty header fields caused by "...,"
    while header and header[-1] == "":
        header.pop()

    n_cols = len(header)
    if n_cols == 0:
        return pd.DataFrame()

    rows = []
    for line in lines[1:]:
        if not line.strip():
            continue

        parts = line.split(",")

        # If row has extra empty trailing fields due to commas at end, truncate
        if len(parts) > n_cols:
            parts = parts[:n_cols]
        # If row is short (rare but happens with malformed lines), pad
        elif len(parts) < n_cols:
            parts = parts + [""] * (n_cols - len(parts))

        rows.append(parts)

    df = pd.DataFrame(rows, columns=header)

    # Cleanup
    df.columns = df.columns.astype(str).str.strip()
    df = df.replace(r"^\s*$", pd.NA, regex=True)

    return df

In [12]:
sensing_root = base_path / "dataset" / "sensing"
sensing_feature_dfs = {}

for modality_folder in sensing_root.iterdir():
    if not modality_folder.is_dir():
        continue

    modality_name = modality_folder.name
    print(f"\nLoading {modality_name}...")

    all_dfs = []

    for file in modality_folder.glob("*.csv"):
        try:
            df = read_sensing_csv(file)

            if df.empty:
                continue

            df["student_id"] = file.stem
            all_dfs.append(df)

        except Exception as e:
            print(f"Skipping {file.name}: {e}")

    if all_dfs:
        sensing_feature_dfs[modality_name] = pd.concat(all_dfs, ignore_index=True)
        print(f"{modality_name} shape:", sensing_feature_dfs[modality_name].shape)
    else:
        print(f"No usable data for {modality_name}")

for name, df in sensing_feature_dfs.items():
    df.columns = df.columns.str.strip()
    sensing_feature_dfs[name] = df


Loading wifi...
wifi shape: (19244309, 5)

Loading gps...
gps shape: (202877, 11)

Loading activity...
activity shape: (22842191, 3)

Loading phonelock...
phonelock shape: (9275, 3)

Loading wifi_location...
wifi_location shape: (1893838, 3)

Loading audio...
audio shape: (99298223, 3)

Loading bluetooth...
bluetooth shape: (1288526, 5)

Loading dark...
dark shape: (7269, 3)

Loading phonecharge...
phonecharge shape: (3318, 3)

Loading conversation...
conversation shape: (79023, 3)


In [13]:
for name, df in sensing_feature_dfs.items():
    print("\n", name)
    print(df.columns)


 wifi
Index(['time', 'BSSID', 'freq', 'level', 'student_id'], dtype='str')

 gps
Index(['time', 'provider', 'network_type', 'accuracy', 'latitude', 'longitude',
       'altitude', 'bearing', 'speed', 'travelstate', 'student_id'],
      dtype='str')

 activity
Index(['timestamp', 'activity inference', 'student_id'], dtype='str')

 phonelock
Index(['start', 'end', 'student_id'], dtype='str')

 wifi_location
Index(['time', 'location', 'student_id'], dtype='str')

 audio
Index(['timestamp', 'audio inference', 'student_id'], dtype='str')

 bluetooth
Index(['time', 'MAC', 'class_id', 'level', 'student_id'], dtype='str')

 dark
Index(['start', 'end', 'student_id'], dtype='str')

 phonecharge
Index(['start', 'end', 'student_id'], dtype='str')

 conversation
Index(['start_timestamp', 'end_timestamp', 'student_id'], dtype='str')


In [14]:
#Testing columns and data types after preprocessing:
for name, df in sensing_feature_dfs.items():
    print(f"\n{name}:")
    print(sensing_feature_dfs[name].head())


wifi:
         time              BSSID  freq  level student_id
0  1364356803  00:0b:86:c1:29:01  2427    -94   wifi_u03
1  1364356803  00:18:01:fb:74:d6  2462    -93   wifi_u03
2  1364356803  00:1d:ce:7b:11:4d  2462    -90   wifi_u03
3  1364356803  00:24:36:a7:61:bd  2427    -92   wifi_u03
4  1364356803  20:aa:4b:cf:33:51  2437    -61   wifi_u03

gps:
         time provider network_type accuracy     latitude    longitude  \
0  1364410654  network         wifi   22.094   43.7066051  -72.2870424   
1  1364411866  network         wifi   24.652   43.7065982  -72.2870054   
2  1364852743  network         wifi    24.06    43.706614  -72.2870392   
3  1364853942  network         wifi   25.242   43.7066032  -72.2870255   
4  1364854001      gps          NaN     14.0  43.70714984  -72.2865919   

        altitude bearing     speed travelstate student_id  
0            0.0     0.0       0.0         NaN    gps_u45  
1            0.0     0.0       0.0         NaN    gps_u45  
2            0.0    

In [15]:

for name, df in sensing_feature_dfs.items():
    if "timestamp" in df.columns:
        time_col = "timestamp"
    elif "time" in df.columns:
        time_col = "time"
    else:
        continue

    df = df.copy()

    # force numeric first (prevents NaT when values are strings like "1364410654")
    t = pd.to_numeric(df[time_col].astype(str).str.strip(), errors="coerce")
    df[time_col] = pd.to_datetime(t, unit="s", errors="coerce")

    # standardize column name
    df["timestamp"] = df[time_col]
    df["date"] = df["timestamp"].dt.date

    sensing_feature_dfs[name] = df

In [16]:
# Start and end timestamp column names:
start_candidates = [
    "start",
    "start_time",
    "start_timestamp",
]

end_candidates = [
    "end",
    "end_time",
    "end_timestamp",
]

def find_column(df, candidates):
    cand_set = {c.strip().lower() for c in candidates}
    for col in df.columns:
        if str(col).strip().lower() in cand_set:
            return col
    return None

def process_interval_df(df):
    start_col = find_column(df, start_candidates)
    end_col = find_column(df, end_candidates)

    if start_col is None or end_col is None:
        print("Start or end column not found.")
        return None

    df = df.copy()

    start_num = pd.to_numeric(df[start_col].astype(str).str.strip(), errors="coerce")
    end_num   = pd.to_numeric(df[end_col].astype(str).str.strip(), errors="coerce")

    df["start_dt"] = pd.to_datetime(start_num, unit="s", errors="coerce")
    df["end_dt"]   = pd.to_datetime(end_num, unit="s", errors="coerce")

    df["duration_s"] = (df["end_dt"] - df["start_dt"]).dt.total_seconds().clip(lower=0)
    df["date"] = df["start_dt"].dt.date

    return df

In [17]:
start_endTimes = ["phonelock", "dark", "phonecharge", "conversation"]

for name in start_endTimes:
    df = sensing_feature_dfs[name]
    df_fixed = process_interval_df(df)

    if df_fixed is not None:
        sensing_feature_dfs[name] = df_fixed 
        print(f"{name} processed and saved.")
    else:
        print(f"{name} could not be processed.")

phonelock processed and saved.
dark processed and saved.
phonecharge processed and saved.
conversation processed and saved.


In [18]:
daily_feature_dfs = []

for name, df in sensing_feature_dfs.items():
    
    if "student_id" not in df.columns or "date" not in df.columns:
        continue

    df = df.copy()

    # If interval-based (has duration_s)
    if "duration_s" in df.columns:
        daily = (
            df.groupby(["student_id", "date"])["duration_s"]
            .sum()
            .reset_index()
        )
        daily = daily.rename(columns={"duration_s": f"{name}_duration_s"})
    
    else:
        # For point-based data, average numeric columns
        numeric_cols = df.select_dtypes(include="number").columns.tolist()
        
        if len(numeric_cols) == 0:
            continue
        
        daily = (
            df.groupby(["student_id", "date"])[numeric_cols]
            .mean()
            .reset_index()
        )
        
        # rename columns to include modality name
        daily = daily.rename(columns={
            col: f"{name}_{col}" for col in numeric_cols
        })
    
    daily_feature_dfs.append(daily)

In [19]:
for k in ["gps", "wifi_location", "phonelock", "dark", "phonecharge", "conversation"]:
    df = sensing_feature_dfs[k]
    print("\n", k)
    print("columns:", df.columns.tolist())
    print("has date?", "date" in df.columns)
    print("has duration_s?", "duration_s" in df.columns)
    print("numeric cols:", df.select_dtypes(include="number").columns.tolist())


 gps
columns: ['time', 'provider', 'network_type', 'accuracy', 'latitude', 'longitude', 'altitude', 'bearing', 'speed', 'travelstate', 'student_id', 'timestamp', 'date']
has date? True
has duration_s? False
numeric cols: []

 wifi_location
columns: ['time', 'location', 'student_id', 'timestamp', 'date']
has date? True
has duration_s? False
numeric cols: []

 phonelock
columns: ['start', 'end', 'student_id', 'start_dt', 'end_dt', 'duration_s', 'date']
has date? True
has duration_s? True
numeric cols: ['start', 'end', 'duration_s']

 dark
columns: ['start', 'end', 'student_id', 'start_dt', 'end_dt', 'duration_s', 'date']
has date? True
has duration_s? True
numeric cols: ['start', 'end', 'duration_s']

 phonecharge
columns: ['start', 'end', 'student_id', 'start_dt', 'end_dt', 'duration_s', 'date']
has date? True
has duration_s? True
numeric cols: ['start', 'end', 'duration_s']

 conversation
columns: ['start_timestamp', 'end_timestamp', 'student_id', 'start_dt', 'end_dt', 'duration_s', '

In [20]:
from functools import reduce

full_daily_df = reduce(
    lambda left, right: pd.merge(
        left, right,
        on=["student_id", "date"],
        how="outer"
    ),
    daily_feature_dfs
)

In [21]:
print(full_daily_df.columns)

Index(['student_id', 'date', 'wifi_freq', 'wifi_level',
       'activity_activity inference', 'phonelock_duration_s',
       'audio_audio inference', 'bluetooth_class_id', 'bluetooth_level',
       'dark_duration_s', 'phonecharge_duration_s', 'conversation_duration_s'],
      dtype='str')


In [48]:
full_daily_df.tail()

,student_id,date,wifi_freq,wifi_level,activity_activity inference,phonelock_duration_s,audio_audio inference,bluetooth_class_id,bluetooth_level,dark_duration_s,phonecharge_duration_s,conversation_duration_s
21029,wifi_u59,2013-05-28,3230.119132,-83.116647,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21030,wifi_u59,2013-05-29,3654.073584,-81.480023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21031,wifi_u59,2013-05-30,3165.509306,-81.583309,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21032,wifi_u59,2013-05-31,3018.856138,-82.912399,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21033,wifi_u59,2013-06-01,3376.851817,-82.884379,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [62]:
import os

def ema_folder_addDate(
    folder_path: str,
    *,
    resp_time_key="resp_time",
    nested_list_keys=("responses", "data", "items"),
    drop_location=True,
    location_name_patterns=("location", "lat", "lon", "latitude", "longitude"),
    epoch_unit="s",   # switch to "ms" if needed
    tz=None,          # e.g., "America/Edmonton"
    debug=False
) -> pd.DataFrame:
    rows = []
    json_paths = []

    # find all jsons anywhere under folder_path
    for root, _, files in os.walk(folder_path):
        for fn in files:
            if fn.lower().endswith(".json"):
                json_paths.append(os.path.join(root, fn))

    if debug:
        print(f"Found {len(json_paths)} json files under: {folder_path}")
        print("Example paths:", json_paths[:3])

    for fp in sorted(json_paths):
        fn = os.path.basename(fp)
        base = os.path.splitext(fn)[0]

        # student_id from filename: take everything after last "_"  (activity_u001 -> u001)
        # if your filenames have multiple underscores, this still works.
        student_id = base.split("_")[-1]

        try:
            with open(fp, "r", encoding="utf-8") as f:
                obj = json.load(f)
        except Exception as e:
            if debug:
                print("Failed to load:", fp, "|", e)
            continue

        # Convert JSON -> list of row dicts
        if isinstance(obj, list):
            items = obj
        elif isinstance(obj, dict):
            list_key = next((k for k in nested_list_keys if k in obj and isinstance(obj[k], list)), None)
            items = obj[list_key] if list_key else [obj]
        else:
            continue

        df = pd.json_normalize(items)
        if df.empty:
            continue

        df.columns = [c.strip() for c in df.columns]
        df["student_id"] = student_id
        df["_source_path"] = fp

        # drop location-ish columns
        if drop_location:
            drop_cols = [c for c in df.columns if any(p in c.lower() for p in location_name_patterns)]
            df = df.drop(columns=drop_cols, errors="ignore")

        # KEEP resp_time as-is, but add date derived from it
        if resp_time_key in df.columns:
            df["resp_time"] = pd.to_numeric(df[resp_time_key], errors="coerce")

            dt = pd.to_datetime(df["resp_time"], errors="coerce", unit=epoch_unit, utc=True)
            if tz is not None:
                dt = dt.dt.tz_convert(tz)
            df["date"] = dt.dt.date

            # if the key wasn't literally "resp_time", drop the original
            if resp_time_key != "resp_time":
                df = df.drop(columns=[resp_time_key], errors="ignore")
        else:
            df["resp_time"] = pd.NA
            df["date"] = pd.NaT

        # convert other columns to numeric if possible (your "1"/"5" strings)
        id_cols = {"student_id", "resp_time", "date", "_source_path"}
        for c in df.columns:
            if c in id_cols:
                continue
            df[c] = pd.to_numeric(df[c], errors="coerce")

        # order columns
        first = ["student_id", "resp_time", "date"]
        rest = [c for c in df.columns if c not in first]
        df = df[first + rest]

        rows.append(df)

    if not rows:
        return pd.DataFrame(columns=["student_id", "resp_time", "date"])

    out = pd.concat(rows, ignore_index=True, sort=False)

    # If duplicates for same (student_id, resp_time), keep first non-null per column
    out = (
        out.sort_values(["student_id", "resp_time"], kind="mergesort")
           .groupby(["student_id", "resp_time"], as_index=False)
           .first()
    )

    return out

In [74]:
EMA_root = base_path / "dataset" / "EMA" / "response"
activity_root = EMA_root / "Activity"

activity_df = ema_folder_addDate(activity_root)
print(activity_df.columns)
print(activity_df.tail())

Index(['student_id', 'resp_time', 'date', 'Social2', 'null', 'other_relaxing',
       'other_working', 'relaxing', 'working', '_source_path'],
      dtype='str')
    student_id   resp_time        date  Social2  null  other_relaxing  \
828        u59  1369792895  2013-05-29      NaN   NaN             1.0   
829        u59  1369930454  2013-05-30      NaN   NaN             3.0   
830        u59  1370023880  2013-05-31      NaN   NaN             5.0   
831        u59  1370208770  2013-06-02      NaN   NaN             3.0   
832        u59  1370303791  2013-06-03      NaN   NaN             2.0   

     other_working  relaxing  working  \
828            5.0       2.0      2.0   
829            3.0       2.0      1.0   
830            1.0       1.0      1.0   
831            3.0       3.0      1.0   
832            3.0       2.0      3.0   

                                          _source_path  
828  /Users/leilatawfik/.cache/kagglehub/datasets/d...  
829  /Users/leilatawfik/.cache/kaggleh

In [75]:
#combining relaxing and other_relaxing into one column:
activity_df["relaxing"] = activity_df[["relaxing", "other_relaxing"]].sum(axis=1)
activity_df = activity_df.drop(columns=["other_relaxing"])

#combining working and other_working into one column:
activity_df["working"] = activity_df[["working", "other_working"]].sum(axis=1)
activity_df = activity_df.drop(columns=["other_working"])

print(activity_df.head())

  student_id   resp_time        date  Social2  null  relaxing  working  \
0        u00  1364437400  2013-03-28      2.0   1.0       0.0      0.0   
1        u00  1364504860  2013-03-28      1.0   3.0       0.0      0.0   
2        u00  1364573072  2013-03-29      1.0   1.0       0.0      0.0   
3        u00  1364590818  2013-03-29      2.0   1.0       0.0      0.0   
4        u00  1364617147  2013-03-30      3.0   2.0       0.0      0.0   

                                        _source_path  
0  /Users/leilatawfik/.cache/kagglehub/datasets/d...  
1  /Users/leilatawfik/.cache/kagglehub/datasets/d...  
2  /Users/leilatawfik/.cache/kagglehub/datasets/d...  
3  /Users/leilatawfik/.cache/kagglehub/datasets/d...  
4  /Users/leilatawfik/.cache/kagglehub/datasets/d...  


In [78]:
#Replace 0.0 with NaN in activity columns to indicate missing data:
activity_cols = ["relaxing", "working"]
for col in activity_cols:
    if col in activity_df.columns:
        activity_df[col] = activity_df[col].replace(0.0, pd.NA)

print(activity_df.tail())

    student_id   resp_time        date  Social2  null relaxing working  \
828        u59  1369792895  2013-05-29      NaN   NaN      3.0     7.0   
829        u59  1369930454  2013-05-30      NaN   NaN      5.0     4.0   
830        u59  1370023880  2013-05-31      NaN   NaN      6.0     2.0   
831        u59  1370208770  2013-06-02      NaN   NaN      6.0     4.0   
832        u59  1370303791  2013-06-03      NaN   NaN      4.0     6.0   

                                          _source_path  
828  /Users/leilatawfik/.cache/kagglehub/datasets/d...  
829  /Users/leilatawfik/.cache/kagglehub/datasets/d...  
830  /Users/leilatawfik/.cache/kagglehub/datasets/d...  
831  /Users/leilatawfik/.cache/kagglehub/datasets/d...  
832  /Users/leilatawfik/.cache/kagglehub/datasets/d...  


In [82]:
sleep_root = EMA_root / "Sleep"

sleep_df = ema_folder_addDate(sleep_root)
print(sleep_df.columns)
print(sleep_df.tail())

Index(['student_id', 'resp_time', 'date', 'null', 'hour', 'rate', 'social',
       '_source_path'],
      dtype='str')
     student_id   resp_time        date  null  hour  rate  social  \
1632        u59  1370152837  2013-06-02   NaN   1.0   3.0     3.0   
1633        u59  1370208645  2013-06-02   NaN  19.0   1.0     1.0   
1634        u59  1370387882  2013-06-04   NaN  13.0   1.0     1.0   
1635        u59  1370477174  2013-06-06   NaN  15.0   1.0     1.0   
1636        u59  1370648801  2013-06-07   NaN   5.0   2.0     3.0   

                                           _source_path  
1632  /Users/leilatawfik/.cache/kagglehub/datasets/d...  
1633  /Users/leilatawfik/.cache/kagglehub/datasets/d...  
1634  /Users/leilatawfik/.cache/kagglehub/datasets/d...  
1635  /Users/leilatawfik/.cache/kagglehub/datasets/d...  
1636  /Users/leilatawfik/.cache/kagglehub/datasets/d...  


In [86]:
EMA_total_df = pd.merge(activity_df, sleep_df, on="student_id")
print(EMA_total_df.columns)
print(EMA_total_df.head())

Index(['student_id', 'resp_time_x', 'date_x', 'null_x', 'hour_x', 'rate_x',
       'social_x', '_source_path_x', 'resp_time_y', 'date_y', 'null_y',
       'hour_y', 'rate_y', 'social_y', '_source_path_y'],
      dtype='str')
  student_id  resp_time_x      date_x  null_x  hour_x  rate_x  social_x  \
0        u00   1364114365  2013-03-24     3.0     NaN     NaN       NaN   
1        u00   1364114365  2013-03-24     3.0     NaN     NaN       NaN   
2        u00   1364114365  2013-03-24     3.0     NaN     NaN       NaN   
3        u00   1364114365  2013-03-24     3.0     NaN     NaN       NaN   
4        u00   1364114365  2013-03-24     3.0     NaN     NaN       NaN   

                                      _source_path_x  resp_time_y      date_y  \
0  /Users/leilatawfik/.cache/kagglehub/datasets/d...   1364114365  2013-03-24   
1  /Users/leilatawfik/.cache/kagglehub/datasets/d...   1364114453  2013-03-24   
2  /Users/leilatawfik/.cache/kagglehub/datasets/d...   1364114760  2013-03-24   
